In [1]:
import os
import json
import mlflow
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from workspace.sources.experiments.metrics import Metric
from workspace.sources.experiments.visualizations.utils import METRICS_PLOT_NAMES_MAPPING
from workspace.sources.local_datasets.dataset import Dataset
from sklearn.metrics import confusion_matrix
from workspace.sources.experiments.metrics import Precision, Loss, standard_evaluation_metrics, standard_metrics

from workspace.sources.experiments.visualizations.plots import plot_confusion_matrix, plot_roc_curve

mlflow.set_tracking_uri('../mlruns')

In [5]:
experiments = ['prefinal-bert-base-uncased', 'prefinal-distillibert-base-uncased']
client = mlflow.tracking.MlflowClient()

### ReCOVery Dataset

In [6]:
dataset_name = 'recovery'

#### Best Models Table

In [7]:
experiments_runs = dict()
for experiment in experiments:
    experiment_runs = client.search_runs(
        experiment_ids=[client.get_experiment_by_name(experiment).experiment_id],
        filter_string=f"params.dataset_name = '{dataset_name}'"
    )
    experiments_runs[experiment] = experiment_runs

In [9]:

best_models_by_metrics = dict()
for metric in standard_evaluation_metrics:
    best_models_by_metrics[metric.name] = dict()
    for experiment, runs in experiments_runs.items():
        best_run = pick_best_run_by_metric(runs, metric)
        if best_run is not None:
            best_models_by_metrics[metric.name][experiment] = best_run


In [34]:


create_metrics_comparison_df(Loss, best_models_by_metrics)

assets/tables/best_models_table_by_loss.csv


,Experiment,Run Signature,Evaluation Metric,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC,FPR,FNR,best_epoch
0,prefinal-bert-base-uncased,"model(mn=bert-base-uncased,mmn=loss,e=10,bs=8,...",loss,bert-base-uncased,0.633333,0.633333,1.0,0.77551,0.722488,1.0,0.0,2.0
1,prefinal-distillibert-base-uncased,"model(mn=distilbert-base-uncased,mmn=loss,e=10...",loss,distilbert-base-uncased,0.666667,0.666667,1.0,0.80000,0.755000,1.0,0.0,2.0


In [35]:



export_best_models_to_latex(best_models_by_loss)


'\\begin{table}[]\n    \\footnotesize\n    \\caption{Best models selected by Loss metric.}\n    \\label{tab:recovery_best_models_by_loss}\n    \\centering\n\\begin{tabular}{|l||r|r|r|r|r|r|r|}\n\\toprule\nModel & Accuracy & \\textbf{Precision} & Recall & \\textbf{F1-Score} & \\textbf{ROC-AUC} & \\textbf{FPR} & FNR \\\\\n\\midrule \\midrule\nbert-base-uncased & 0.633 & 0.633 & 1.000 & 0.776 & 0.722 & 1.000 & 0.000 \\\\\ndistilbert-base-uncased & 0.667 & 0.667 & 1.000 & 0.800 & 0.755 & 1.000 & 0.000 \\\\\n\\bottomrule\n\\end{tabular}\n\\end{table}'